# Futevôlei: jugadas y estadísticas desde el video

**Paso 1**: detecta las jugadas y genera el video cortado (solo jugadas, numeradas) para revisarlo.

**Revisión**: mirando ese video se anotan jugadas falsas, faltantes y los cambios de lado.

**Paso 2**: estadísticas (jugadas, saques y ataques con su resultado, resumen por equipo) y video con las cajas de pelota y jugadores.

**Paso 3**: subir las estadísticas a sql.


## 0. Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU")

### Código del proyecto
Subir la carpeta con los modulos a Drive y poner en `CODE_DIR` la carpeta que la **contiene**.

También se puede subir un .zip a `CODE_DIR`: la celda siguiente lo descomprime como `futevolei_code/` (si hay zip, siempre reemplaza a la carpeta).

In [ ]:
import sys, os, zipfile, shutil, tempfile
CODE_DIR = "/content/drive/MyDrive"            # carpeta que contiene futevolei_code/ (o futevolei.zip)
PKG_NAME = "futevolei_code"                    # nombre de la carpeta del paquete
pkg_dir  = os.path.join(CODE_DIR, PKG_NAME)
zip_path = os.path.join(CODE_DIR, "futevolei.zip")

if os.path.exists(zip_path):
    # si hay zip, manda el zip: se descomprime y reemplaza a la carpeta del paquete
    tmp = tempfile.mkdtemp()
    zipfile.ZipFile(zip_path).extractall(tmp)
    shutil.rmtree(pkg_dir, ignore_errors=True)
    shutil.copytree(os.path.join(tmp, "futevolei"), pkg_dir)
assert os.path.exists(os.path.join(pkg_dir, "__init__.py")), f"Falta la carpeta {PKG_NAME}/ o futevolei.zip en {CODE_DIR}"
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# borrar los módulos ya importados: así toma los cambios si se actualizan los .py en Drive
for name in [m for m in sys.modules if m == PKG_NAME or m.startswith(PKG_NAME + ".")]:
    del sys.modules[name]
from futevolei_code import stage1_detect, stage2_statistics
import pandas as pd
from IPython.display import display

### Configuración

In [ ]:
from ultralytics import YOLO

VIDEO      = "/content/drive/MyDrive/KC_Proyecto/Futevolei_Dataset/videos_crudos/ESTRÉIA EM GRANDE ESTILO - JAMEL E DAVI X DUDU E ARTHUR #futevoleibrasil #cfb - FUTEVÔLEI BRASIL (1080p, h264, youtube).mp4"    # video a procesar
OUT_FOLDER = "/content/salida_video_test"                                 # carpeta de resultados

ball_player_model = YOLO("/content/drive/MyDrive/KC_Proyecto/ball_player_yolov8s_best.pt")   # pelota + jugadores
court_kp_model    = YOLO("/content/drive/MyDrive/KC_Proyecto/court_yolo11_large_.pt")        # keypoints de cancha
print(ball_player_model.names)   # confirmar: 0 = jugador, 1 = pelota

## Paso 1: jugadas y video cortado
`stage1_detect`: encontrar las jugadas

Recibe: el video y el modelo de pelota y jugadores.

- Detecciones (track_play_events): corre YOLO en cada frame, a la resolución de entrenamiento (1280), y guarda las cajas de pelota y jugadores con su confianza. Es el paso lento y el que más usa la GPU.

- Limpieza:
remove_static_balls saca las pelotas que no se mueven, como las de repuesto en la arena.
remove_static_objects saca objetos fijos que el modelo confunde con la pelota, como el borde superior de la red.

- Trayectoria (build_ball_trajectory): elige una posición de la pelota por frame, rellena huecos cortos y suaviza para sacar velocidades. Funciona como un seguimiento propio de la pelota.
- Repeticiones (detect_replay_graphics + replay_mask_from_graphics): recorre el video buscando la animación amarilla que abre y cierra cada repetición, y arma la máscara de frames a ignorar.

- Jugadas (detect_rallies):
Inicio: detecta el saque: pelota quieta, patada desde los pies de un jugador, vista abierta, sube, avanza y pasa tiempo en el aire.
Fin: decide el final: la pelota toca la arena, sale del cuadro, se pierde o la transmisión corta a un primer plano.

- Material de revisión:
review_video.mp4: solo las jugadas, cada una con su número.
review.csv: minuto de cada jugada en el video original y en el de revisión.

- Guarda: detections.parquet, meta.csv, replay_graphics.csv, rallies_stage1.csv, review.csv y review_video.mp4.

In [ ]:
e1 = stage1_detect(VIDEO, ball_player_model, out_dir=OUT_FOLDER)
e1.keys()

In [ ]:
display(e1["review"])
print("Video para revisar:", e1["review_video"])

## Revisión manual

Mirando `review_video.mp4` (cada jugada tiene su número arriba a la izquierda):

- `remove`: números de las jugadas que NO son jugadas (repeticiones, pases, etc.)
- `add`: jugadas que faltan, como `("mm:ss", "mm:ss")` de inicio y fin en el **video original**
- `side_switches`: número de la **primera** jugada con los equipos cambiados de lado (o `"mm:ss"` del video original si esa jugada es una agregada)
- `names`: `"Equipo 1"` es el equipo que saca la jugada 1, si no se ponen los nombres se guardaran como `"Equipo"` 1 y `"Equipo 2"`

In [ ]:
remove        = []
add           = []
side_switches = [7, 13, 19, 25, 31, 37]
names         = {"Equipo 1": "DUDU E ARTHUR", "Equipo 2": "DAVI E JAMEL"}

## Paso 2: estadísticas y video con cajas
`stage2_statistics`: estadísticas y video con cajas

- Recibe: el video, lo de la etapa 1 (o None para cargarlo del disco), el modelo de la cancha y lo anotado en la revisión.

- Aplica la revisión: elimina y agrega jugadas, las renumera y traduce los cambios de lado a la numeración nueva.

- Movimiento de cámara (camera_motion): lee el video dentro de las jugadas y mide cuánto gira la cámara en cada frame y dónde hay cortes. Con eso se sabe dónde está la red aunque no se vea.

- Homografías (compute_homographies_in_rallies): corre el modelo de puntos de la cancha en las jugadas y calcula la transformación de la imagen a la cancha (1800 × 900).

- Estadísticas (compute_statistics):
Red en cada frame: la ubica con la homografía y el giro de la cámara.

- Lado de la pelota y cruces: de qué lado de la red está la pelota y cuándo cruza.
Toques: cambios bruscos de velocidad que no explica la gravedad.

- Posesiones y golpes: con eso arma las posesiones de cada equipo y los golpes que cruzan la red: el saque y los ataques.
Ganador de cada jugada: dónde cayó la pelota, si se vio con homografía; si no, el equipo que saca la jugada siguiente.

- Resultado de cada golpe: punto, defendido o fuera.

- Equipos: cada lado se asigna a Equipo 1 o 2 según los cambios de lado anotados.

- Video con cajas (draw_detections): solo las jugadas, con las cajas de pelota y jugadores, el punto de la trayectoria y la línea de la red.

- Guarda: rallies.csv, cam_motion.parquet, homographies.pkl, rally_stats.csv, shots.csv, summary.csv y rallies_video_boxes.mp4.

In [ ]:
e2 = stage2_statistics(VIDEO, e1, court_kp_model, out_dir=OUT_FOLDER,
                       remove=remove, add=add,
                       side_switches=side_switches, names=names)

e2.keys()

In [ ]:
e2["summary"]

In [ ]:
print("Resumen por equipo")
display(e2["summary"])
print("Jugadas")
display(e2["rally_stats"][["rally_id", "serving_team", "serve_result", "n_attacks",
                       "attacks_team_1", "attacks_team_2", "ends_with",
                       "final_result", "winner_team", "winner_method"]])
print("Saques y ataques")
display(e2["shots"])
print("Video con cajas:", e2["video_boxes"])

## Guardar en la base de datos (PostgreSQL)

Guarda `videos`, `teams`, `rallies` (= `e2["rally_stats"]`) y `statistics` (= `e2["summary"]`).

- La base tiene que ser accesible desde internet: Supabase, Neon, Render, etc.
- Si el video ya estaba guardado (mismo `VIDEO_NAME`), se reemplaza.


In [ ]:
!pip -q install psycopg2-binary
import os, pandas as pd
from google.colab import userdata
from futevolei_code.database import connect, save_to_sql

conn = connect(host="aws-0-us-west-2.pooler.supabase.com",          
               dbname="postgres",
               user="postgres.zjemgccvehasnvtmvdex",
               password= "*****",
               port=6543,
               sslmode="require")


meta = pd.read_csv(os.path.join(OUT_FOLDER, "meta.csv"))
VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO))[0]   
video_id = save_to_sql(conn, e2, meta, video_name=VIDEO_NAME, source_path=VIDEO,
                       model_version="ball_player_yolov8s_best")
conn.close()